# AI Use Statement
Claude and the Antigravity IDE (integrated development environment) chatbot were used for code generation and explaining topics. 

Claude gave me the steps and code for pixi set, repo set up.

**AI USE IS NOTED IN THE CODE AND MARKDOWN CELLS.**

I attempted agentic coding but for the most part it gave cryptic looking code that is not human-readable. I had to explicitly and repeatedly ask for clearer and simpler code. I provided the guidelines so that it can also review everything. The end result is a mixture between agentic and much back and forth. 

The first cell is a copy from checkpoint 1. Claude generated it after I went through the reasons for each variable and decided on how to address extreme values.

Pluses: It is very fast. It can be guided and repeated attempts and expected to output much more readable code.

I note how I used it through the notebook.

Claude did final reviews to correct header formatting, table formatting, and define previously undefined variables.

In [ ]:
# Packages:
from pathlib import Path
import subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
ROOT = Path.cwd()
DATA = ROOT / "data"
OUT = ROOT / "output"
OUT.mkdir(exist_ok=True)

# 2.1 Modeling table

This wide set is from checkpoint 1. It is in the gitignore so it does not show up on github. I have just copied over and pasted. After looking at each variable and the extremes, I used to Claude help create the rule set and create the wide table in readable way.

There is one row per entry or stay for simplicity.

In [ ]:
DESCRIPTORS = ["Age", "Gender", "Height", "ICUType", "Weight"]

def load_set_a(data_dir):
    files = sorted(data_dir.glob("set-a/*.txt"))
    df = pd.concat([pd.read_csv(f).assign(RecordID=int(f.stem)) for f in files], ignore_index=True)
    df["Hours"] = pd.to_timedelta(df["Time"] + ":00").dt.total_seconds() / 3600
    is_static = (df["Hours"] == 0) & df["Parameter"].isin(DESCRIPTORS + ["RecordID"])
    static = df[is_static].pivot(index="RecordID", columns="Parameter", values="Value")[DESCRIPTORS]
    return df[~is_static], static

long, static = load_set_a(DATA)
#long.to_csv(OUT / "set-a_long.csv", index=False)

LIMITS = {  # inclusive physically-possible range; any parameter not listed is a lab value and only needs to be > 0
    "SysABP": (1, 300), "DiasABP": (1, 300), "MAP": (1, 300),          # 0 = line disconnected; 300 = transducer ceiling
    "NISysABP": (1, 300), "NIDiasABP": (1, 300), "NIMAP": (1, 300),    # 0 = cuff failed to read
    "HR": (0, 300), "RespRate": (0, 80), "Urine": (0, np.inf),          # 0 = asystole / apnea / anuria
    "Temp": (13, 47), "pH": (6.5, 8), "PaCO2": (5, 250), "PaO2": (10, 700), "FiO2": (0.21, 1),
    "SaO2": (0, 100), "HCT": (0, 100), "GCS": (3, 15), "MechVent": (1, 1), "Albumin": (0, 7), "Weight": (5, 500),
    "Age": (15, 120), "Gender": (0, 1), "ICUType": (1, 4), "Height": (50, 275),   # descriptors; -1 = unknown
}

def clean(rows):
    p, v = rows["Parameter"], rows["Value"].copy()
    v = v.mask((p == "pH") & (v > 100), v / 100)            # 733 -> 7.33: decimal point lost
    v = v.mask((p == "Height") & (v > 275), v / 2.54)       # 431.8 -> 170: cm typed into an inches field
    v = v.mask((p == "Height") & (v < 3), v * 100)          # 1.8 -> 180: metres instead of cm
    repaired = v != rows["Value"]

    lo = p.map(lambda x: LIMITS.get(x, (0, np.inf))[0])
    hi = p.map(lambda x: LIMITS.get(x, (0, np.inf))[1])
    out_of_range = (v < lo) | (v > hi) | (~p.isin(LIMITS) & (v == 0))
    v = v.mask(out_of_range)

    last_beat = rows.loc[(p == "HR") & (v > 0)].groupby("RecordID")["Hours"].max()
    hr_artefact = (p == "HR") & (v == 0) & (rows["Hours"] < rows["RecordID"].map(last_beat))   # HR 0 then beats again = lead off

    bp = rows.assign(Value=v)[p.str.endswith("ABP")].pivot_table(index=["RecordID", "Time"], columns="Parameter", values="Value")
    key = pd.MultiIndex.from_frame(rows[["RecordID", "Time"]])
    dia_ge_sys = pd.Series(False, index=rows.index)
    for s, d in [("SysABP", "DiasABP"), ("NISysABP", "NIDiasABP")]:
        dia_ge_sys |= key.isin(bp.index[bp[d] >= bp[s]]) & p.isin([s, d])
    v = v.mask(hr_artefact | dia_ge_sys)

    reason = pd.Series("", index=rows.index)
    reason[repaired] = "unit/decimal repaired"
    reason[out_of_range] = "outside possible range"
    reason[hr_artefact] = "HR 0 with later heartbeat"
    reason[dia_ge_sys] = "diastolic >= systolic"
    return rows.assign(Value=v), rows.assign(NewValue=v, Reason=reason)[reason != ""]

static_rows = static.stack().rename("Value").reset_index().assign(Time="00:00", Hours=0.0)
clean_all, changed = clean(pd.concat([long, static_rows], ignore_index=True))
is_static = (clean_all["Hours"] == 0) & clean_all["Parameter"].isin(DESCRIPTORS)
long_clean = clean_all[~is_static]
static_clean = clean_all[is_static].pivot(index="RecordID", columns="Parameter", values="Value")[DESCRIPTORS]


valid = long_clean.dropna(subset=["Value"])

outcomes = pd.read_csv(DATA / "Outcomes-a.txt").set_index("RecordID")
STATS = ["count", "first", "last", "min", "max", "mean"]

summary = valid.groupby(["RecordID", "Parameter"])["Value"].agg(STATS).unstack("Parameter")
summary["count"] = summary["count"].fillna(0).astype(int)          # never measured = 0, not NaN
summary.columns = [p + "_" + s for s, p in summary.columns]         # (mean, HR) -> HR_mean
wide = static_clean.join(summary).join(outcomes)
# wide.to_csv(OUT / "set-a_wide.csv")
print(wide.shape)
wide.head()

Here, we can see that there are 4000 rows / unique Record IDs / number of patients.

In [ ]:
wide.index.is_unique

In [ ]:
in_hospital_deaths = wide["In-hospital_death"]
in_hospital_deaths.value_counts()

In [ ]:
in_hospital_deaths.value_counts(1)

This is our outcome. We can see that there were 554 deaths out of the 4000, which is about 14%. Today I learned that value_counts(1) returns proportions.

In [ ]:
benchmark_scores = wide[["SAPS-I", "SOFA"]]

benchmark_scores.describe()

We see that the benchmark scores contain -1. So, we will have to set them -1 to NaN (not a number, meaning missing).

In [ ]:
benchmark_scores = wide[["SAPS-I", "SOFA"]].replace(-1, np.nan)
benchmark_scores.isna().sum()


There are 190 and 144 NaN values for SAPS-I (Simplified Acute Physiology Score I) and SOFA (Sequential Organ Failure Assessment), respectively.

# 2.2 Predictors

Here, we have to set up the predictors. We need to remove variables like SAPS-I, SOFA, Survival, and Length_of_stay.

Survival and Length_of_stay were dropped because they happened after hour 48. Including them would be leakage.
Every predictor uses 0 to 48 hours only.
There is a 20% threshold. If variables were only available on 20% of stays then they are dropped but their counts are kept.

In [ ]:
excluded_predictors = ["SAPS-I", "SOFA", "In-hospital_death", "Survival", "Length_of_stay"]
# drop columns that are not predictors
predictors = wide.drop(columns=excluded_predictors)
predictors.head()

Set ICUType to categorical.

In [ ]:
predictors["ICUType"] = predictors["ICUType"].astype("category")

AI Use: I asked for help in determining which variables to drop. It suggested MecnVent first/last/min/max/mean columns because MechVent is always 1 when record. Similarly for Cholestrol and Troponin I, because they are rarely measured. We keep the count columns for those variables.

In [ ]:
coverage = (predictors.filter(like="_count") > 0).mean()
coverage.sort_values().head()

We can see that the coverage is low for TroponinI_count, Cholesterol_count.

In [ ]:
predictors = predictors.drop(columns=["MechVent_first", "MechVent_last", "MechVent_min", "MechVent_max", "MechVent_mean",
                                      "Cholesterol_first", "Cholesterol_last", "Cholesterol_min", "Cholesterol_max", "Cholesterol_mean",
                                      "TroponinI_first", "TroponinI_last", "TroponinI_min", "TroponinI_max", "TroponinI_mean"])
predictors.shape

Now, we can look at groups of data. I had Claude help write this line of code to show the values for each group.

In [ ]:
pd.Series(predictors.columns.str.split("_").str[-1]).value_counts()

Table requested from section 2.2 Predictors:

| Group | Example columns | Count | Reason |
|---|---|---|---|
| Descriptors | Age, Gender, Height, ICUType, Weight | 5 | Useful demographics and intensive care unit (ICU) type, which are known at admission |
| First/initial value | HR_first, BUN_first | 34 | Patients' condition on arrival at ICU, which is important for predicting outcomes |
| Last/final value | HR_last, BUN_last | 34 | The patient's state at the end of the 48-hour window, which is also important for predicting outcomes |
| Minimum value | HR_min, Temp_min | 34 | The lowest values |
| Maximum value | HR_max, Temp_max | 34 | The worst high values, such as a racing heart or high lactate |
| Mean value | HR_mean, Temp_mean | 34 | Average over 48 hours or stay |
| Number of measurements | HR_count, Temp_count | 37 | More measurements may mean that the patient is in worse condition. |
| **Total** | | **212** | |

HR = heart rate, BUN = blood urea nitrogen, Temp = temperature.

# 2.3 Cross-validation design

Models will use a 5-fold cross-validation design. 80% of the data used for training and 20% for testing. 

In [ ]:
# packages:
from sklearn.model_selection import StratifiedKFold
from sklearn.base import clone
from sklearn.metrics import roc_auc_score

In [ ]:
# set seed
SEED = 1

Here, we set up the cross-validation design. We set a random seed for reproducibility.

In [ ]:
# cv
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

# get the folds
folds = list(cv.split(predictors, in_hospital_deaths))

# inner cv
inner_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

Here, I got help from Claude and ChatGPT to write the cross-validation model. 

**AI USE CONS:** It takes multiple tries to output readable code that is easy to follow. For example, it initially ouputted this code: 

def run_cv(model, X, y):
    oof = np.zeros(len(y))
    aucs = []
    fitted = []
    for train, test in folds:
        m = clone(model).fit(X.iloc[train], y.iloc[train])
        oof[test] = m.predict_proba(X.iloc[test])[:, 1]
        aucs.append(roc_auc_score(y.iloc[test], oof[test]))
        fitted.append(m)
    return oof, aucs, fitted

I had a hard time understanding the code and what it was doing. So, I had to do keep asking for clarification using ChatGPT to translate code into a more readable format. Below this the code:


In [ ]:
def run_cross_validation(model, features, target, folds):
    "Train one model per fold and collect validation predictions."

    # One prediction per original row
    out_of_fold_predictions = np.zeros(len(target))

    # area under the ROC curve for each validation fold
    validation_auc_scores = []

    # this is the trained model from each fold
    trained_models = []

    for training_rows, validation_rows in folds:
        
        # Train on this fold's training rows
        fold_model = clone(model)
        fold_model.fit(
            features.iloc[training_rows],
            target.iloc[training_rows],
        )

        # predict probabilities for rows excluded from training
        validation_probabilities = fold_model.predict_proba(
            features.iloc[validation_rows]
        )[:, 1]

        # put predictions back into their original row positions
        out_of_fold_predictions[validation_rows] = validation_probabilities

        # measure of this fold's performance.
        fold_auc = roc_auc_score(
            target.iloc[validation_rows],
            validation_probabilities,
        )
        validation_auc_scores.append(fold_auc)

        # we then keep the model for later use
        trained_models.append(fold_model)
    # trained_models is every model trained on each fold
    # out_of_fold_predictions is the predicted probabilities for each row in the original dataset
    # validation_auc_scores is the AUC score for each validation fold
    return out_of_fold_predictions, validation_auc_scores, trained_models


Imputation and scaling are fit inseach training fold. Tuning uses nested cross validation.

# 2.4 LASSO logistic regression

In [ ]:
# packages:
from sklearn.compose import make_column_transformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegressionCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

LASSO (least absolute shrinkage and selection operator) sets coefficients to zero for variables that are not useful for prediction. 

**AI USE:**
I asked both Claude and ChatGPT to help me write the code for LASSO logistic regression. I then asked for further clarification. I added comments that represent my interpretation of the code.

In [ ]:
# we have to set ICUType as a categorical variable and all other variables as numeric variables

NUMERIC_COLUMNS = predictors.columns.drop("ICUType")
CATEGORICAL_COLUMNS = ["ICUType"]


# here we are rescaleing the numeric variables to have a mean of 0 and a standard deviation of 1. We are also imputing missing values with the median value for each variable. 

numeric_preprocessing = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
)



preprocess = make_column_transformer(
    (numeric_preprocessing, NUMERIC_COLUMNS),
    (
        OneHotEncoder(drop="first", handle_unknown="ignore"),
        CATEGORICAL_COLUMNS,
    ),
    verbose_feature_names_out=False,
)

lasso_model = LogisticRegressionCV(
    
    # this tries 13 penalty strengths (C values) logarithmically spaced between 0.001 and 1.
    
    Cs=np.logspace(-3, 0, 13),
    
    cv=inner_cv,
    
    # uses l1 penealty, which is the LASSO penalty.
    
    l1_ratios=[1.0],
    use_legacy_attributes=False,
    solver="liblinear",
    scoring="roc_auc",
    max_iter=5_000,
    
    # use same seed for reproducibility
    
    random_state=SEED,
)

lasso = make_pipeline(
    preprocess,
    lasso_model,
)

Here, we do the cross-validation for the LASSO logistic regression model.

In [ ]:
out_of_fold_lasso, area_under_curve_lasso, lasso_folds = run_cross_validation(lasso, predictors, in_hospital_deaths, folds)

In [ ]:
print(area_under_curve_lasso)
np.mean(area_under_curve_lasso)

We need to determine the penalty strength (C) for each fold. 

In [ ]:
chosen_penalties = []
for fold_model in lasso_folds:
    logistic_step = fold_model[-1]
    chosen_penalties.append(logistic_step.C_)

np.round(chosen_penalties, 3)

Here, we refit the model on all of the data. 

In [ ]:
lasso.fit(predictors, in_hospital_deaths)

column_names = lasso[0].get_feature_names_out()
logistic_step = lasso[-1]
coefficients = pd.Series(logistic_step.coef_[0], index=column_names)

kept_coefficients = coefficients[coefficients != 0]
kept_coefficients = kept_coefficients.sort_values()

len(kept_coefficients)

Based on the below chart, we can see that the top 3 predictors for greater odds of death are Age, pH_mean, and GCS_min (minimum Glasgow Coma Scale, GCS) in that order.

GCS_last has the largest magnitude and is a negative coefficient, meaning that higher GCS_last is associated with lower odds of death.

In [ ]:
kept_coefficients.plot.barh(figsize=(10, 18))
plt.xlabel("Change in log-odds of death per 1 SD")
plt.title("LASSO coefficients, refit on all of set-a")
plt.show()

Here are the values of the kept coefficients, top 5 by magnitude.

In [ ]:
# output the values of the kept coefficients, top 5 by magnitude
kept_coefficients.abs().sort_values(ascending=False).head(5)


AI USE: I asked Claude to write code below.

I asked for the number dropped variables. We can see that 71 were dropped in all folds and 37 were kept across all folds.

In [ ]:
kept_in_each_fold = []
for fold_model in lasso_folds:
    logistic_step = fold_model[-1]
    kept_in_each_fold.append(logistic_step.coef_[0] != 0)

kept = pd.DataFrame(kept_in_each_fold, columns=column_names)
folds_kept = kept.sum()
folds_kept.value_counts().sort_index()

These are the predictors that kept across all folds. We can see that there are 37 predictors. LASSO picks from these 37 no matter the which patients it trains on. The other 106 were kept in only 1 to 4 folds.

In [ ]:
always_kept = folds_kept[folds_kept == 5].index
always_kept

# 2.5 GAM

In [ ]:
# packages
from pygam import LogisticGAM, s, f

AI USE CONS: Claude provided an arbitrary list of predictors to use for the generalized additive model (GAM) with no actual reason why.  Here is the original list: "gam_columns = [
    "Age", "GCS_min", "HR_mean", "Temp_mean", "NIMAP_min", "PaO2_min", "BUN_last",
    "Creatinine_max", "Urine_mean", "HCO3_last", "Lactate_max", "Na_last",
    "Platelets_min", "WBC_max",
]"

After questioning it, it outputted an the 14 SAPS-I predictors with further recommendations:

gam_columns = [
    "Age", "HR_mean", "NISysABP_mean", "Temp_mean", "MechVent_count",
    "Urine_mean", "BUN_mean", "HCT_mean", "WBC_mean", "Glucose_mean",
    "K_mean", "Na_mean", "HCO3_mean", "GCS_min",]

NISysABP = non-invasive (cuff) systolic blood pressure, SysABP = arterial (invasive) systolic blood pressure, MechVent = mechanical ventilation, HCT = hematocrit, WBC = white blood cell count, K = potassium, Na = sodium, HCO3 = bicarbonate.

This use of the SAPS-I predictors makes more sense. SAPS-I uses systolic blood pressure but here, we use the cuff pressure NISysABP_mean instead of SysABP_mean because the it shows that the arterial is missing for 31% of the stays while cuff pressure is only missing for about 13% of stays. It also uses MechVent_count instead of the respiratory rate because the respiratory rate is missing for 73% of the stays.

It also uses GCS_min because low GCS is a greater risk, hence more informative.

In [ ]:
gam_columns = [
    "Age", "HR_mean", "NISysABP_mean", "Temp_mean", "MechVent_count",
    "Urine_mean", "BUN_mean", "HCT_mean", "WBC_mean", "Glucose_mean",
    "K_mean", "Na_mean", "HCO3_mean", "GCS_min",
]

gam_predictors = predictors[gam_columns].copy()
gam_predictors["ICUType"] = predictors["ICUType"].cat.codes

There are 14 predictors in the GAM model.

In [ ]:
# here we get the number of smooth terms in the GAM model. 
number_of_smooth_terms = len(gam_columns)

gam_terms = s(0)
for column_number in range(1, number_of_smooth_terms):
    gam_terms = gam_terms + s(column_number)
gam_terms = gam_terms + f(number_of_smooth_terms)

print(number_of_smooth_terms)
gam_terms

AI USE: I asked Claude to write code below.

This is the cross-validation for the GAM model. It uses gridsearch to find the best smoothing parameter for each fold. It uses the same folds as the LASSO model.

In [ ]:
np.seterr(all="ignore")

out_of_fold_gam = np.zeros(len(in_hospital_deaths))
area_under_curve_gam = []

for training_rows, validation_rows in folds:
    imputer = SimpleImputer(strategy="median")
    imputer.fit(gam_predictors.iloc[training_rows])

    training_features = imputer.transform(gam_predictors.iloc[training_rows])
    validation_features = imputer.transform(gam_predictors.iloc[validation_rows])

    gam = LogisticGAM(gam_terms)
    gam.gridsearch(training_features, in_hospital_deaths.iloc[training_rows], progress=False)

    validation_probabilities = gam.predict_proba(validation_features)
    out_of_fold_gam[validation_rows] = validation_probabilities

    fold_auc = roc_auc_score(in_hospital_deaths.iloc[validation_rows], validation_probabilities)
    area_under_curve_gam.append(fold_auc)

np.round(area_under_curve_gam, 3)

We then refit the model on all of the data.

In [ ]:
final_imputer = SimpleImputer(strategy="median")
final_imputer.fit(gam_predictors)
all_gam_features = final_imputer.transform(gam_predictors)

final_gam = LogisticGAM(gam_terms)
final_gam.gridsearch(all_gam_features, in_hospital_deaths, progress=False)
final_gam.summary()

AI USE CON: Claude keeps suggesting arbitrary predictors to plot and not one based on the coefficients. I had to go back ask it to plot all of the predictors instead and does a poor job or interpreting plots.

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(20, 10))

for ax, column in zip(axes.flat, gam_columns):
    term_number = gam_columns.index(column)
    grid = final_gam.generate_X_grid(term=term_number)
    effect, confidence_band = final_gam.partial_dependence(term=term_number, X=grid, width=0.95)

    column_values = grid[:, term_number]
    ax.plot(column_values, effect)
    ax.fill_between(column_values, confidence_band[:, 0], confidence_band[:, 1], alpha=0.3)
    ax.set_xlabel(column)
    ax.set_ylabel("Log-odds of death")

axes.flat[-1].axis("off")
plt.tight_layout()
plt.show()

We can see below that the predictor with the greatest curvature is BUN_mean. Age appears to have the most linear relationship with the outcome. Greater the age, the greater the odds of death. Mean sodium, mean bicarbonate, and mean temperature have very similar U-shapred curvatures. There is a negative trend for mean Potassium. As K+ levels increase, the odds of death decrease. 

# 2.6 Gradient boosting

Claude did the gradient boost set up. Here, GridSearchCV is to find the best hyperparameters for the model. There are 16 combinations of hyperparameters to test. The best hyperparameters are then used to fit the model on the training data.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GridSearchCV

boosting_grid = {
    "learning_rate": [0.03, 0.1],
    "max_depth": [3, 6],
    "min_samples_leaf": [20, 60],
    "l2_regularization": [0, 1],
}

boosting_model = HistGradientBoostingClassifier(
    max_iter=1000,
    early_stopping=True,
    categorical_features="from_dtype",
    random_state=SEED,
)

boosting = GridSearchCV(
    boosting_model,
    boosting_grid,
    cv=inner_cv,
    scoring="roc_auc",
    n_jobs=-1,
)

This is the cross-validation for the gradient boosting model. It uses the same folds as the LASSO model.

In [ ]:
out_of_fold_boosting, area_under_curve_boosting, boosting_folds = run_cross_validation(
    boosting, predictors, in_hospital_deaths, folds
)
np.round(area_under_curve_boosting, 3)

One row per fold

In [ ]:
chosen_settings = []
for fold_model in boosting_folds:
    chosen_settings.append(fold_model.best_params_)

pd.DataFrame(chosen_settings)

The grid and tuning happens inside the training folds. Missing values are handled by the model and not imputed. Early stopping uses part of the training fold, never the held-out fold.

# 2.7 Comparison and final model

Code was generated with Claude

Here, missing scores are filled with teh training fold median. Logistic regression turns the score into a probability.

In [ ]:
from sklearn.linear_model import LogisticRegression

score_model = make_pipeline(
    SimpleImputer(strategy="median"),
    LogisticRegression(),
)

out_of_fold_saps, area_under_curve_saps, saps_folds = run_cross_validation(
    score_model, benchmark_scores[["SAPS-I"]], in_hospital_deaths, folds
)
out_of_fold_sofa, area_under_curve_sofa, sofa_folds = run_cross_validation(
    score_model, benchmark_scores[["SOFA"]], in_hospital_deaths, folds
)

This is our comparison table. Rows 0 to 4 are the folds.

In [ ]:
auc_table = pd.DataFrame({
    "LASSO": area_under_curve_lasso,
    "GAM": area_under_curve_gam,
    "Boosting": area_under_curve_boosting,
    "SAPS-I": area_under_curve_saps,
    "SOFA": area_under_curve_sofa,
})

auc_summary = auc_table.agg(["mean", "std"])
pd.concat([auc_table, auc_summary]).round(3)

This shows the best model minus each benchmark.

In [ ]:
boosting_minus_saps = auc_table["Boosting"] - auc_table["SAPS-I"]
boosting_minus_saps.round(3)

In [ ]:
boosting_minus_sofa = auc_table["Boosting"] - auc_table["SOFA"]
boosting_minus_sofa.round(3)

All five differences are positive, which means that the model performs better than the benchmark. This means that boosting beats SOFA by 0.211 to 0.259, which is significantly larger than the between fold variation. The same can be said about boosting and the SAPS-I benchmark.

Here, this shows out-of-fold predictions.

In [ ]:
out_of_fold_predictions = pd.DataFrame({
    "died": in_hospital_deaths,
    "lasso": out_of_fold_lasso,
    "gam": out_of_fold_gam,
    "boosting": out_of_fold_boosting,
    "saps1": out_of_fold_saps,
    "sofa": out_of_fold_sofa,
})

out_of_fold_predictions.head()

In [ ]:
from sklearn.metrics import RocCurveDisplay

model_columns = ["lasso", "gam", "boosting", "saps1", "sofa"]
fig, ax = plt.subplots(figsize=(6, 6))

for column in model_columns:
    RocCurveDisplay.from_predictions(
        out_of_fold_predictions["died"],
        out_of_fold_predictions[column],
        name=column,
        ax=ax,
    )

ax.plot([0, 1], [0, 1], linestyle="--", color="grey")
plt.show()

This figure shows the receiver operating characteristic (ROC) curves from the out of fold predictions. Gradient boosting (area under the curve, AUC 0.85) and LASSO (AUC 0.84) performed similary and much better than GAM (AUC 0.80).

I asked Claude for more specificity and specific for the performace at different false positive rates.

In [ ]:
from sklearn.metrics import roc_curve

sensitivity_table = {}
for column in model_columns:
    false_positive_rate, true_positive_rate, thresholds = roc_curve(
        out_of_fold_predictions["died"],
        out_of_fold_predictions[column],
    )
    sensitivity_table[column] = np.interp([0.10, 0.20, 0.30], false_positive_rate, true_positive_rate)

pd.DataFrame(sensitivity_table, index=["FPR 10%", "FPR 20%", "FPR 30%"]).round(2)

We can see at a false positive rate of 20%, boosting identified 74% of deaths, which is better than other two models and relatively much better than the SAPS-1 and SOFA benchmarks, which were 36% and 35%, respectively.

Here, we refit the model on all of set-a.

In [ ]:
boosting.fit(predictors, in_hospital_deaths)
boosting.best_params_

I chose to fit boosting because of a slightly better performance.

In [ ]:
import joblib

final_model = boosting.best_estimator_
joblib.dump(final_model, OUT / "cp2_final_model.joblib")
out_of_fold_predictions.to_csv(OUT / "cp2_oof_predictions.csv")